In [ ]:
%%capture
import os
import math

from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
igh_folder = Path(os.environ["IGH_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)


In [ ]:
from django_pandas.io import read_frame
from meta_subject.models import HivExitReview, SubjectVisit
from edc_visit_schedule.models import SubjectScheduleHistory
from meta_lists.models import ArvRegimens
from meta_screening.models import SubjectScreening


In [ ]:
df_hiv = read_frame(HivExitReview.objects.all(), verbose=False).rename(columns={"subject_visit": "subject_visit_id", "id": "hiv_exit_review_id"})
regimens = read_frame(ArvRegimens.objects.all(), verbose=False)
df_hiv = df_hiv.merge(regimens[["id", "name"]], left_on="current_arv_regimen", right_on="id", how="left").rename(columns={"name": "current_arv_regimen_name"})
df_subject_visit = read_frame(SubjectVisit.objects.all(), verbose=False).rename(columns={"id": "subject_visit_id"})
df_subject_screen = read_frame(SubjectScreening.objects.values("subject_identifier", "hospital_identifier").all(), verbose=False)
df_onschedule = read_frame(SubjectScheduleHistory.objects.all(), verbose=False)
df_hiv = df_hiv.merge(
    df_subject_visit[["subject_visit_id", "subject_identifier", "visit_code", "visit_code_sequence"]], on="subject_visit_id", how="left")

df = df_onschedule[["subject_identifier", "onschedule_datetime", 'onschedule_model', "offschedule_datetime", "site"]].merge(df_hiv, on="subject_identifier", how="left", suffixes=["", "_y"]).drop(columns=["site_y"])

df = df.merge(df_subject_screen[["subject_identifier", "hospital_identifier"]], on="subject_identifier", how="left")

In [ ]:
df

In [ ]:
# those on schedule 'schedule'
df[(df["hiv_exit_review_id"].isna()) & (df["onschedule_model"]=="meta_prn.onschedule") & (df["offschedule_datetime"].isna())][["site", "subject_identifier", "hospital_identifier", "offschedule_datetime"]]


In [ ]:
# those off schedule 'schedule'
df[(df["hiv_exit_review_id"].isna()) & (df["onschedule_model"]=="meta_prn.onschedule") & ~(df["offschedule_datetime"].isna())][["site", "subject_identifier", "hospital_identifier", "offschedule_datetime"]]


In [ ]:
# those on schedule other than 'schedule', e.g. preg or DM
df[(df["hiv_exit_review_id"].isna()) & ~(df["onschedule_model"].isin(["meta_prn.onschedule"])) & (df["offschedule_datetime"].isna())][["site", "subject_identifier", "hospital_identifier", "offschedule_datetime"]]

In [ ]:

df[~df["current_arv_regimen_name"].isna()][["site", "subject_identifier", "hospital_identifier", "offschedule_datetime", "current_arv_regimen_name", "other_current_arv_regimen"]]

In [ ]:
df_hiv2 = read_frame(HivExitReview.objects.all(), verbose=False)
regimens = read_frame(ArvRegimens.objects.all(), verbose=False)
df_hiv2 = df_hiv2.merge(regimens[["id", "name"]], left_on="current_arv_regimen", right_on="id", how="left").rename(columns={"name": "current_arv_regimen_name"})

In [ ]:
df_hiv2

In [ ]:
df_hiv2[["singleton_field", "current_arv_regimen", "other_current_arv_regimen"]]


In [ ]:
df_hiv